# Glutamate response analysis run

Use this notebook to load one session, run activation/tuning/sequence analyses, and save standardized output tables.

In [ ]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    resolve_glutamate_analysis_paths,
    run_glutamate_analysis,
)

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

In [ ]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [ ]:
assets[0]

In [ ]:
for asset in assets[:]:
    session_root = asset.session_dir
    paths = resolve_glutamate_analysis_paths(session_root)
    print(paths)
    
    config = GlutamateAnalysisConfig(
    alpha=0.05,
    min_events_activation=8,
    min_events_tuning_per_image=5,
    min_images_for_tuning=4,
    min_images_for_sequence=4,
    min_positions_for_sequence=3,
    n_shuffles_tuning=1000,
    random_seed=0,
    )
    config
    
    results = run_glutamate_analysis(session_root, config=config)
    {k: v.shape for k, v in results.items() if hasattr(v, 'shape')}
    
    activation_summary = results['activation_summary_table']
    tuning_summary = results['tuning_summary_table']
    sequence_summary = results['sequence_summary_table']

    display(activation_summary.head())
    display(tuning_summary.head())
    display(sequence_summary.head())
    
    output_dir = resolve_glutamate_analysis_paths(session_root).output_dir
    print(output_dir)
    print(sorted(p.name for p in output_dir.iterdir()))